In [1]:
import torch 
from torch import nn 
import math

In [34]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, seq_len=100):
        super().__init__()
        pe = torch.zeros((seq_len, d_model))
        position = torch.arange(seq_len).unsqueeze(1)
        # Standard formula: 10000^(2i/d_model)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term) # Fixed slice
        self.register_buffer('pe', pe.unsqueeze(0)) # Shape (1, seq_len, d_model)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, n_heads, d_model):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, d_model * 3)
        self.out_proj = nn.Linear(d_model, d_model)
        self.scale = math.sqrt(self.d_head)

    def forward(self, x):
        B, L, D = x.shape
        qkv = self.qkv(x).view(B, L, 3, self.n_heads, self.d_head).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) / self.scale
        mask = torch.tril(torch.ones(L, L, device=x.device))
        attn = attn.masked_fill(mask == 0, float("-inf"))
        attn = torch.softmax(attn, dim=-1)
        
        out = (attn @ v).transpose(1, 2).reshape(B, L, D)
        return self.out_proj(out)

class TransformerBlock(nn.Module):
    def __init__(self, n_heads, d_model):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(n_heads, d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.ReLU(),
            nn.Linear(d_model * 4, d_model)
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class Transformer(nn.Module): 
    def __init__(self, n_heads, vocab_size, d_model, n_layers, seq_len=100):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, seq_len)
        self.blocks = nn.ModuleList([TransformerBlock(n_heads, d_model) for _ in range(n_layers)])
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x = self.positional_encoding(x)
        for block in self.blocks:
            x = block(x)
        return self.lm_head(x)

In [ ]:
VOCAB_SIZE = 1000
D_MODEL = 128
N_HEADS = 8
N_LAYERS = 4
SEQ_LEN = 20
BATCH_SIZE = 2

Success!
Input shape: torch.Size([2, 20])
Output shape: torch.Size([2, 20, 1000])


In [38]:

model = Transformer(N_HEADS, VOCAB_SIZE, D_MODEL, N_LAYERS, SEQ_LEN)

inp = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN))

try:
    output = model(inp)
    print("Success!")
    print(f"Input shape: {inp.shape}")  # [2, 20]
    print(f"Output shape: {output.shape}")    # [2, 20, 1000]
except Exception as e:
    print(f"Error: {e}")

Success!
Input shape: torch.Size([2, 20])
Output shape: torch.Size([2, 20, 1000])


In [39]:
# Our "Dataset"
text = "abcdefghijklmnopqrstuvwxyz"
chars = sorted(list(set(text)))
vocab_size = len(chars)

# Mappings
char_to_int = {ch: i for i, ch in enumerate(chars)}
int_to_char = {i: ch for i, ch in enumerate(chars)}

def encode(s): return torch.tensor([char_to_int[c] for c in s], dtype=torch.long).unsqueeze(0)
def decode(l): return ''.join([int_to_char[i.item()] for i in l])

print(f"Vocab size: {vocab_size}")

Vocab size: 26


In [40]:
import torch.optim as optim

# Initialize model with our vocab size
model = Transformer(n_heads=4, vocab_size=vocab_size, d_model=64, n_layers=2, seq_len=len(text))
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Prepare data: Input "abcdefg...", Target "bcdefgh..."
input_seq = encode(text[:-1]) # "abcdef...y"
target_seq = encode(text[1:])  # "bcdefg...z"

# Simple training loop
model.train()
for epoch in range(200):
    optimizer.zero_grad()
    
    output = model(input_seq) # Shape: [1, seq_len, vocab_size]
    
    # Reshape for CrossEntropy: (Batch * Seq, Vocab)
    loss = criterion(output.view(-1, vocab_size), target_seq.view(-1))
    
    loss.backward()
    optimizer.step()
    
    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 3.6828
Epoch 50, Loss: 0.0036
Epoch 100, Loss: 0.0015
Epoch 150, Loss: 0.0010


In [41]:
model.eval()
start_char = "a"
input_test = encode(start_char)
generated = start_char

with torch.no_grad():
    for _ in range(5): # Predict the next 5 characters
        logits = model(input_test)
        last_token_logits = logits[0, -1, :] # Get the last predicted character
        next_token_id = torch.argmax(last_token_logits).item()
        
        generated += int_to_char[next_token_id]
        # Append predicted token to input for next step
        input_test = torch.cat([input_test, torch.tensor([[next_token_id]])], dim=1)

print(f"Model Input: 'a' -> Model Output: '{generated}'")

Model Input: 'a' -> Model Output: 'abcdef'
